# 07 - Sistemas não lineares
Vamos aprender sobre como usar o método iterativo de Newton-Raphson adaptado aos sistemas de equações, envolvendo aproximações iniciais, cálculo da matriz do Jacobiano e critério de parada, para resolução de sistemas não lineares.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana7
```

Faça o checkout nessa nova branch:

```bash
git checkout semana7
```

<hr />

## Atividade 1
Escreva o método iterativo de Newton-Raphson adaptado dentro do arquivo $utils/algoritmos.py$.

In [ ]:
import numpy as np
from scipy.optimize import approx_fprime


def JN(x, F, eps=1e-8):
    x = np.asarray(x, dtype=float)
    n = x.size
    Jnum = np.zeros((n, n), dtype=float)

    def fi(v):
        return F(v)[i]

    for i in range(n):
        Jnum[i, :] = approx_fprime(x, fi, epsilon=eps)

    return Jnum


def G(x, F, J):
    return x - np.linalg.inv(J(x)) @ F(x)


def GN(x, F):
    return x - np.linalg.inv(JN(x, F)) @ F(x)


def fixed_point(a, g, TOL=1e-8, iter=1000):
    x = g(a)
    i = 1
    while np.linalg.norm(x - a) > TOL and i < iter:
        a = x
        x = g(a)
        i += 1
    return x

Encontre uma aproximação numérica para o seguinte problema não linear de três equações e três incógnitas:

$$
2x_{1} - x_{2} = \cos(x_{1})
$$

$$
- x_{1} + 2x_{2} - x_{3} = \cos(x_{2})
$$

$$
- x_{2} + x_{3} = \cos(x_{3})
$$

Partindo da seguinte aproximação inicial:

$$
x^{(0)} = (1,\,1,\,1)^{T}
$$

In [2]:
import sys
import os
import numpy as np

# Sobe um nível para a raiz do projeto e adiciona ao caminho do sistema
sys.path.append(os.path.abspath(os.path.join('..')))

from utils.algoritmos import (
    G,
    GN,
    fixed_point,
)


def main():

    def F(x):
        x1, x2, x3 = x
        return np.array(
            [
                2.0 * x1 - x2 - np.cos(x1),
                -x1 + 2.0 * x2 - x3 - np.cos(x2),
                -x2 + x3 - np.cos(x3),
            ],
            dtype=float,
        )

    def J(x):
        x1, x2, x3 = x
        return np.array(
            [
                [2.0 + np.sin(x1), -1.0, 0.0],
                [-1.0, 2.0 + np.sin(x2), -1.0],
                [0.0, -1.0, 1.0 + np.sin(x3)],
            ],
            dtype=float,
        )

    x = np.array([1.0, 1.0, 1.0], dtype=float)
    r = fixed_point(x, lambda x: G(x, F, J))
    print(r)
    r = fixed_point(x, lambda x: GN(x, F))
    print(r)


if __name__ == "__main__":
    main()

[0.95127255 1.32189759 1.44618584]
[0.95127255 1.32189759 1.44618584]


## Atividade 2
Encontre uma aproximação em cada incógnita para a solução próxima da origem do sistema:

$$
6x - 2y + e^{z} = 2
$$

$$
\sin(x) - y + z = 0
$$

$$
\sin(x) + 2y + 3z = 1
$$

**Resposta esperada:**

$$
x \approx 0.259751, \quad y \approx 0.302736, \quad z \approx 0.045896
$$

In [1]:
import numpy as np



def F(variaveis):

    x, y, z = variaveis

    return np.array([

        6 * x - 2 * y + np.exp(z) - 2,

        np.sin(x) - y + z,

        np.sin(x) + 2 * y + 3 * z - 1,

    ])



def jacobiano(variaveis):

    x, y, z = variaveis

    return np.array([

        [6, -2, np.exp(z)],

        [np.cos(x), -1, 1],

        [np.cos(x), 2, 3],

    ])



solucao = np.zeros(3)

TOL = 1e-12

MAX_ITER = 50



for iteracao in range(1, MAX_ITER + 1):

    passo = np.linalg.solve(jacobiano(solucao), -F(solucao))

    solucao += passo

    if np.linalg.norm(passo, ord=np.inf) < TOL:

        break

else:

    raise RuntimeError("O método de Newton não convergiu.")



print(f"Solução: x = {solucao[0]:.6f}, y = {solucao[1]:.6f}, z = {solucao[2]:.6f}")

print(f"Iterações: {iteracao}")

print(f"Norma do resíduo: {np.linalg.norm(F(solucao), ord=np.inf):.2e}")

Solução: x = 0.259751, y = 0.302736, z = 0.045896
Iterações: 4
Norma do resíduo: 2.22e-16


## Atividade 3
Considere o problema de encontrar os pontos de interseção das curvas descritas:

$$
\frac{x^{2}}{8} + \frac{(y-1)^{2}}{5} = 1
$$

$$
\tan^{-1}(x) + x = y + y^{3}
$$

Com base no gráfico, encontre soluções aproximadas para o problema e use-as para iniciar o método de Newton-Raphson para encontrar as raízes.

**Resposta esperada:**

$$
(-1.2085435,\,-1.0216674) 
\quad \text{e} \quad 
(2.7871115,\,1.3807962)
$$

In [2]:
import numpy as np



def F(ponto):

    x, y = ponto

    return np.array([

        x**2 / 8 + (y - 1)**2 / 5 - 1,

        np.arctan(x) + x - y - y**3,

    ])



def jacobiano(ponto):

    x, y = ponto

    return np.array([

        [x / 4, 2 * (y - 1) / 5],

        [1 / (1 + x**2) + 1, -1 - 3 * y**2],

    ])



def newton_sistema(chute, tol=1e-12, max_iter=50):

    ponto = np.array(chute, dtype=float)

    for iteracao in range(1, max_iter + 1):

        passo = np.linalg.solve(jacobiano(ponto), F(ponto))

        ponto -= passo

        if np.linalg.norm(passo, ord=np.inf) < tol:

            return ponto, iteracao

    raise RuntimeError("O método de Newton não convergiu.")



chutes_iniciais = [(-1.2, -1.0), (2.8, 1.4)]

for chute in chutes_iniciais:

    raiz, iteracoes = newton_sistema(chute)

    residuo = np.linalg.norm(F(raiz), ord=np.inf)

    print(

        f"Chute {chute}: (x, y) = ({raiz[0]:.7f}, {raiz[1]:.7f}); "

        f"iterações = {iteracoes}; resíduo = {residuo:.2e}"

    )

Chute (-1.2, -1.0): (x, y) = (-1.2085435, -1.0216674); iterações = 4; resíduo = 4.44e-16
Chute (2.8, 1.4): (x, y) = (2.7871115, 1.3807962); iterações = 4; resíduo = 4.44e-16


## Atividade 4
Uma indústria consome energia elétrica de três usinas fornecedoras.  

O custo de fornecimento em reais por hora como função da potência consumida em kW é dada pelas seguintes funções:

$$
C_{1}(x) = 10 + 0.3x + 10^{-4}x^{2} + 3.4 \cdot 10^{-9}x^{4} \tag{5.61}
$$

$$
C_{2}(x) = 50 + 0.25x + 2 \cdot 10^{-4}x^{2} + 4.3 \cdot 10^{-7}x^{3}
$$

$$
C_{3}(x) = 500 + 0.19x + 5 \cdot 10^{-4}x^{2} + 1.1 \cdot 10^{-7}x^{4}
$$

Calcule a distribuição de consumo que produz custo mínimo quando a potência total consumida é **1500 kW**.  

Denote por $x_{1}, x_{2}, x_{3}$ as potências consumidas das usinas 1, 2 e 3, respectivamente.  

O custo total será dado por:

$$
C(x_{1},x_{2},x_{3}) = C_{1}(x_{1}) + C_{2}(x_{2}) + C_{3}(x_{3})
$$

Enquanto o consumo total é:

$$
G(x_{1},x_{2},x_{3}) = x_{1} + x_{2} + x_{3} - 1500 = 0
$$

Pelos multiplicadores de Lagrange, temos que resolver o sistema:

$$
\nabla C(x_{1},x_{2},x_{3}) = \lambda \nabla G(x_{1},x_{2},x_{3})
$$

$$
G(x_{1},x_{2},x_{3}) = 0
$$

**Resposta esperada:**

$$
(x_{1},x_{2},x_{3}) \approx (453.62, \; 901.94, \; 144.43)
$$

In [3]:
import numpy as np



def custos_marginais(x):

    x1, x2, x3 = x

    return np.array([

        0.3 + 2e-4 * x1 + 13.6e-9 * x1**3,

        0.25 + 4e-4 * x2 + 1.29e-6 * x2**2,

        0.19 + 1e-3 * x3 + 4.4e-7 * x3**3,

    ])



def hessiana_custos(x):

    x1, x2, x3 = x

    return np.diag([

        2e-4 + 40.8e-9 * x1**2,

        4e-4 + 2.58e-6 * x2,

        1e-3 + 1.32e-6 * x3**2,

    ])



def sistema_lagrange(variaveis):

    x = variaveis[:3]

    multiplicador = variaveis[3]

    return np.append(custos_marginais(x) - multiplicador, np.sum(x) - 1500)



def jacobiano_lagrange(variaveis):

    jacobiano = np.zeros((4, 4))

    jacobiano[:3, :3] = hessiana_custos(variaveis[:3])

    jacobiano[:3, 3] = -1

    jacobiano[3, :3] = 1

    return jacobiano



solucao = np.array([800.0, 400.0, 300.0, 0.5])

TOL = 1e-10

MAX_ITER = 50



for iteracao in range(1, MAX_ITER + 1):

    passo = np.linalg.solve(jacobiano_lagrange(solucao), sistema_lagrange(solucao))

    solucao -= passo

    if np.linalg.norm(passo, ord=np.inf) < TOL:

        break

else:

    raise RuntimeError("Newton-Raphson não convergiu.")



x1, x2, x3 = solucao[:3]

custo_total = (

    10 + 0.3 * x1 + 1e-4 * x1**2 + 3.4e-9 * x1**4

    + 50 + 0.25 * x2 + 2e-4 * x2**2 + 4.3e-7 * x2**3

    + 500 + 0.19 * x3 + 5e-4 * x3**2 + 1.1e-7 * x3**4

)



print(f"Distribuição ótima: x1={x1:.4f} kW, x2={x2:.4f} kW, x3={x3:.4f} kW")

print(f"Potência total: {x1 + x2 + x3:.10f} kW")

print(f"Multiplicador de Lagrange: lambda={solucao[3]:.8f}")

print(f"Custo total: R$ {custo_total:.4f}/h")

print(f"Norma do resíduo do sistema: {np.linalg.norm(sistema_lagrange(solucao), ord=np.inf):.2e}")



esperado = np.array([453.62, 901.94, 144.43])

print(f"Custos marginais na resposta esperada: {custos_marginais(esperado)}")

Distribuição ótima: x1=453.6224 kW, x2=901.9432 kW, x3=144.4344 kW
Potência total: 1500.0000000000 kW
Multiplicador de Lagrange: lambda=1.66019422
Custo total: R$ 1650.0645/h
Norma do resíduo do sistema: 0.00e+00
Custos marginais na resposta esperada: [1.66017368 1.66018554 1.66006789]


## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 7"
git push origin semana7
```